In [1]:
import sys
import os
sys.path.append(os.pardir)

In [2]:
import pandas as pd

In [3]:
from config import GRIDSTATUS_API_KEY

In [4]:
from gridstatusio import GridStatusClient

# Load Data From Gridstatus API

## Basic test example from the docs
- free plan has a 1 million rows per month limit
- add a 'limit' to all get_dataset request to avoid blowing through the limit

In [5]:
#Test
client = GridStatusClient(api_key = GRIDSTATUS_API_KEY)

In [ ]:
#returns a pandas df
data = client.get_dataset('ercot_fuel_mix', limit=100, start='2025-01-01', end='2025-01-02')

In [10]:
data.head()

,interval_start_utc,interval_end_utc,coal_and_lignite,hydro,nuclear,power_storage,solar,wind,natural_gas,other
0,2025-01-01 00:00:00+00:00,2025-01-01 00:05:00+00:00,9777.7,224.4,5089.8,3456.3,0.4,4392.2,25303.8,0.0
1,2025-01-01 00:05:00+00:00,2025-01-01 00:10:00+00:00,9778.6,224.5,5091.5,3288.6,0.4,4528.6,25249.4,0.0
2,2025-01-01 00:10:00+00:00,2025-01-01 00:15:00+00:00,9787.5,223.8,5092.6,3204.9,0.4,4668.8,25178.9,0.0
3,2025-01-01 00:15:00+00:00,2025-01-01 00:20:00+00:00,9834.4,223.4,5089.3,2910.9,0.5,4782.5,25268.9,0.0
4,2025-01-01 00:20:00+00:00,2025-01-01 00:25:00+00:00,9843.1,223.6,5090.1,2833.7,0.4,4915.6,25254.0,0.0


## Checking the API usage

In [11]:
usage = client.get_api_usage()

In [13]:
usage

{'plan_name': 'Free',
 'limits': {'api_rows_returned_limit': 500000,
  'api_requests_limit': 250,
  'api_rows_per_response_limit': 50000,
  'per_second_api_rate_limit': 1,
  'per_minute_api_rate_limit': 30,
  'per_hour_api_rate_limit': 600},
 'current_usage_period_start': '2026-05-01T00:00:00Z',
 'current_usage_period_end': '2026-06-01T00:00:00Z',
 'current_period_usage': {'total_requests': 1, 'total_api_rows_returned': 100}}

### Retry logic for heavy pagination
- client will retry from rate limit error 429 and server errors 5XX

In [ ]:
client = GridStatusClient(
    max_retries=3,        # Maximum retries (default: 5)
    base_delay=1.0,       # Base delay in seconds (default: 2.0)
    exponential_base=1.5, # Exponential backoff multiplier (default: 2.0)
)

In [14]:
### Pulling spp load
spp_data = client.get_dataset('spp_load_hourly', limit=100, start='2025-01-01', end='2025-01-02')

2026-05-30 14:02:37 - INFO - Fetching Page 1...
2026-05-30 14:02:37 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-05-30 14:02:37 - INFO - Params: {'start_time': Timestamp('2025-01-01 00:00:00'), 'end_time': Timestamp('2025-01-02 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 100, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-05-30 14:02:37 - INFO - Done in 0.48 seconds. 
2026-05-30 14:02:37 - INFO - Total rows: 100/100 (100.0% of limit)
2026-05-30 14:02:37 - INFO - Total number of rows: 100


In [15]:
spp_data.head()

,interval_start_utc,interval_end_utc,balancing_area_name,control_zone_name,forecast_area_type,load
0,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,CSWS,CF,5315.603
1,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,EDE,CF,665.349
2,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,GRDA,CF,870.240
3,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,INDN,CF,125.553
4,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,KACY,CF,273.514


### Filtering the data
- balancing_area_name has SPP, SWPW and another?
-  The Areas in this dataset are the legacy balancing authorities for the region that SPP serves
-  so filter the "balancing_are_name" to just "SPP"
-  what is the 'control_zone_name'? I think we need to sum across all the control zones to get the total system SPP region load
-  timestamp is "interval_start_utc"

In [9]:
spp_data = client.get_dataset(
    'spp_load_hourly',  
    start='2025-01-01', 
    end='2025-01-02',
    columns = ["interval_start_utc", "balancing_area_name", "control_zone_name", "forecast_area_type", "load"],
    filter_column = "balancing_area_name",
    filter_value = "SPP",
    limit=200
)

2026-06-01 14:15:34 - INFO - Fetching Page 1...
2026-06-01 14:15:34 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-06-01 14:15:34 - INFO - Params: {'start_time': Timestamp('2025-01-01 00:00:00'), 'end_time': Timestamp('2025-01-02 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 200, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'balancing_area_name', 'filter_value': 'SPP', 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-06-01 14:15:35 - INFO - Done in 0.51 seconds. 
2026-06-01 14:15:35 - INFO - Total rows: 200/200 (100.0% of limit)
2026-06-01 14:15:35 - INFO - Total number of rows: 200


In [10]:
spp_data.head()

,interval_start_utc,interval_end_utc,balancing_area_name,control_zone_name,forecast_area_type,load
0,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,CSWS,CF,5315.603
1,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,EDE,CF,665.349
2,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,GRDA,CF,870.240
3,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,INDN,CF,125.553
4,2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+00:00,SPP,KACY,CF,273.514


In [11]:
spp_data.balancing_area_name.unique()

<StringArray>
['SPP']
Length: 1, dtype: str

In [12]:
spp_data.forecast_area_type.unique()

<StringArray>
['CF']
Length: 1, dtype: str

## Transform Load

In [ ]:
## Filter out for NC load
filter_data = spp_data[spp_data["forecast_area_type"] == "CF"]

In [22]:
#Aggregate accros control zones
total_load = spp_data.groupby("interval_start_utc", as_index = False).agg(load = ("load", "sum"))

In [23]:
total_load.head()

,interval_start_utc,load
0,2025-01-01 00:00:00+00:00,67730.086
1,2025-01-01 01:00:00+00:00,66692.826
2,2025-01-01 02:00:00+00:00,65610.642
3,2025-01-01 03:00:00+00:00,64622.860
4,2025-01-01 04:00:00+00:00,63448.364


In [24]:
total_load.dtypes

interval_start_utc    datetime64[us, UTC]
load                              float64
dtype: object

In [25]:
total_load.index

RangeIndex(start=0, stop=12, step=1)

In [26]:
#set the datetime index and frequency
total_load.set_index("interval_start_utc", inplace = True)
total_load.index.freq = 'h'

In [27]:
total_load.head()

,load
interval_start_utc,
2025-01-01 00:00:00+00:00,67730.086
2025-01-01 01:00:00+00:00,66692.826
2025-01-01 02:00:00+00:00,65610.642
2025-01-01 03:00:00+00:00,64622.860
2025-01-01 04:00:00+00:00,63448.364


In [28]:
total_load.index

DatetimeIndex(['2025-01-01 00:00:00+00:00', '2025-01-01 01:00:00+00:00',
               '2025-01-01 02:00:00+00:00', '2025-01-01 03:00:00+00:00',
               '2025-01-01 04:00:00+00:00', '2025-01-01 05:00:00+00:00',
               '2025-01-01 06:00:00+00:00', '2025-01-01 07:00:00+00:00',
               '2025-01-01 08:00:00+00:00', '2025-01-01 09:00:00+00:00',
               '2025-01-01 10:00:00+00:00', '2025-01-01 11:00:00+00:00'],
              dtype='datetime64[us, UTC]', name='interval_start_utc', freq=None)

In [ ]:
#time zone is us, UTC
datetime64(precision, [timezone])
'2025-01-01 00:00:00+00:00'
'2025-01-01 00:00:00+00:00'

# Pulling Historical Temp Forecast
- OKC: 35.481918, -97.508469
- KC: 39.099724, -94.578331
- Sioux Falls: 43.536388, -96.731667

In [5]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [6]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [32]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 39.0997,
	"longitude": -94.5786,
	"start_date": "2025-01-01",
	"end_date": "2025-01-02",
	"hourly": "temperature_2m",
}

In [33]:
responses = openmeteo.weather_api(url, params = params)

In [34]:
response = responses[0]

In [20]:
#print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")

Coordinates: 39.10891342163086°N -94.58583068847656°E


In [35]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

In [36]:
hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

In [31]:
#hourly_data

{'date': DatetimeIndex(['2025-01-01 00:00:00+00:00', '2025-01-01 01:00:00+00:00',
                '2025-01-01 02:00:00+00:00', '2025-01-01 03:00:00+00:00',
                '2025-01-01 04:00:00+00:00', '2025-01-01 05:00:00+00:00',
                '2025-01-01 06:00:00+00:00', '2025-01-01 07:00:00+00:00',
                '2025-01-01 08:00:00+00:00', '2025-01-01 09:00:00+00:00',
                '2025-01-01 10:00:00+00:00', '2025-01-01 11:00:00+00:00',
                '2025-01-01 12:00:00+00:00', '2025-01-01 13:00:00+00:00',
                '2025-01-01 14:00:00+00:00', '2025-01-01 15:00:00+00:00',
                '2025-01-01 16:00:00+00:00', '2025-01-01 17:00:00+00:00',
                '2025-01-01 18:00:00+00:00', '2025-01-01 19:00:00+00:00',
                '2025-01-01 20:00:00+00:00', '2025-01-01 21:00:00+00:00',
                '2025-01-01 22:00:00+00:00', '2025-01-01 23:00:00+00:00',
                '2025-01-02 00:00:00+00:00', '2025-01-02 01:00:00+00:00',
                '2025-01-02 02

In [37]:
hourly_data["temperature_2m"] = hourly_temperature_2m

In [38]:
hourly_dataframe = pd.DataFrame(data = hourly_data)

In [39]:
hourly_dataframe.head()

,date,temperature_2m
0,2025-01-01 00:00:00+00:00,-0.1385
1,2025-01-01 01:00:00+00:00,-0.1385
2,2025-01-01 02:00:00+00:00,-0.1385
3,2025-01-01 03:00:00+00:00,-0.3885
4,2025-01-01 04:00:00+00:00,-0.2885
